In [1]:
import numpy as np
import pandas as pd
import scipy
from scipy import stats
import datetime as dt
import dask.dataframe as dd

import matplotlib.pyplot as plt
from matplotlib import colors
import soundfile as sf
import matplotlib.patches as patches
from pathlib import Path
from sklearn.cluster import KMeans
import fsspec

In [2]:
import sys

sys.path.append("../src")
sys.path.append("../src/bout")

In [3]:
from core import SITE_NAMES, FREQUENCY_COLOR_MAPPINGS
import clustering as clstr
import bout as bt
import plot as bt_plt
import activity.subsampling as ss
import activity.activity_assembly as actvt

from cli import get_file_paths
from calls import plot_call_features, compute_features, call_extraction

['/Users/adityakrishna/duty-cycle-investigation/daily_notebook', '/Users/adityakrishna/miniconda3/envs/dc-study/lib/python311.zip', '/Users/adityakrishna/miniconda3/envs/dc-study/lib/python3.11', '/Users/adityakrishna/miniconda3/envs/dc-study/lib/python3.11/lib-dynload', '', '/Users/adityakrishna/miniconda3/envs/dc-study/lib/python3.11/site-packages', '../src', '../src/bout', '../src', '../src/bout', '../src', '../src', '../src', '../src', '../src/activity', '/Users/adityakrishna/duty-cycle-investigation/daily_notebook/../src/calls', '/Users/adityakrishna/duty-cycle-investigation/daily_notebook/../src', '/Users/adityakrishna/duty-cycle-investigation/daily_notebook/../src/bout', '/Users/adityakrishna/duty-cycle-investigation/daily_notebook/../src']


In [4]:
FREQUENCY_COLOR_MAPPINGS = {
                    'LF' : 'cyan',
                    'HF' : 'orange'
                        }

LABEL_FOR_GROUPS = {
                    0: 'LF', 
                    1: 'HF'
                    }

FIGSIZE = (12, 6)

DURATION = 300

In [5]:
PADDED_CALL_LENGTH = 0.06

def open_and_get_call_info(audio_file, dets):
    welch_key = 'all_locations'
    output_dir = Path(f'../data/generated_welch/{welch_key}')
    output_file_type = 'top1_inbouts_welch_signals'
    welch_data = pd.read_csv(output_dir / f'2022_{welch_key}_{output_file_type}.csv', index_col=0, low_memory=False)
    k = 2
    kmean_welch = KMeans(n_clusters=k, n_init=10, random_state=1).fit(welch_data.values)

    features_of_interest = gather_features_of_interest(dets, kmean_welch, audio_file)

    dets.reset_index(drop=True, inplace=True)
    dets['index'] = dets.index
    dets['file_name'] = pd.DatetimeIndex(pd.to_datetime(dets['input_file'], format='%Y%m%d_%H%M%S', exact=False)).strftime('%Y%m%d_%H%M%S.WAV')
    dets['sampling_rate'] = len(dets) * [audio_file.samplerate]
    dets.insert(0, 'SNR', features_of_interest['snrs'])
    dets.insert(0, 'peak_frequency', features_of_interest['peak_freqs'])
    dets.insert(0, 'KMEANS_CLASSES', pd.Series(features_of_interest['classes']).map(LABEL_FOR_GROUPS))

    return features_of_interest['call_signals'], dets

def gather_features_of_interest(dets, kmean_welch, audio_file):
    fs = audio_file.samplerate
    features_of_interest = dict()
    features_of_interest['call_signals'] = []
    features_of_interest['welch_signals'] = []
    features_of_interest['snrs'] = []
    features_of_interest['peak_freqs'] = []
    features_of_interest['classes'] = []
    nyquist = fs//2
    for index, row in dets.iterrows():
        audio_seg, length_of_section = get_section_of_call_in_file(row, audio_file)
        
        freq_pad = 2000
        low_freq_cutoff = row['low_freq']-freq_pad
        high_freq_cutoff = min(nyquist-1, row['high_freq']+freq_pad)
        band_limited_audio_seg = call_extraction.bandpass_audio_signal(audio_seg, fs, low_freq_cutoff, high_freq_cutoff)

        signal = band_limited_audio_seg.copy()
        signal[:int(fs*(length_of_section))] = 0
        noise = band_limited_audio_seg - signal
        snr_call_signal = signal[-int(fs*length_of_section):]
        snr_noise_signal = noise[:int(fs*length_of_section)]
        features_of_interest['call_signals'].append(snr_call_signal)

        snr = call_extraction.get_snr_from_band_limited_signal(snr_call_signal, snr_noise_signal)
        features_of_interest['snrs'].append(snr)

        welch_info = dict()
        welch_info['num_points'] = 100
        max_visible_frequency = 96000
        welch_info['max_freq_visible'] = max_visible_frequency
        welch_signal = compute_features.compute_welch_psd_of_call(snr_call_signal, fs, welch_info)
        features_of_interest['welch_signals'].append(welch_signal)

        peaks = np.where(welch_signal==max(welch_signal))[0][0]
        features_of_interest['peak_freqs'].append((max_visible_frequency/len(welch_signal))*peaks)
        
        welch_signal = (welch_signal).reshape(1, len(welch_signal))
        features_of_interest['classes'].append(kmean_welch.predict(welch_signal)[0])

    features_of_interest['call_signals'] = np.array(features_of_interest['call_signals'], dtype='object')

    return features_of_interest

def get_section_of_call_in_file(detection, audio_file):
    fs = audio_file.samplerate
    call_dur = (detection['end_time'] - detection['start_time'])
    pad = 0.004
    start = detection['start_time'] - call_dur - (3*pad)
    duration = (2 * call_dur) + (4*pad)
    end = detection['end_time']
    audio_file.seek(int(fs*start))
    audio_seg = audio_file.read(int(fs*duration))

    length_of_section = call_dur + (2*pad)

    return audio_seg, length_of_section

In [6]:
def plot_colored_dets_over_audio_seg(audio_features, spec_features, plot_dets):
    """
    Function to plot the spectrogram of a provided audio segment with overlayed detections
    """

    audio_seg = audio_features['audio_seg']
    fs = audio_features['sample_rate']
    start = audio_features['start']
    duration = audio_features['duration']

    plt.figure(figsize=FIGSIZE)
    plt.rcParams.update({'font.size': 24})
    plt.title(f"BatDetect2 detections on {audio_features['file_path'].name}", fontsize=22)
    plt.specgram(audio_seg, NFFT=spec_features['NFFT'], cmap=spec_features['cmap'], vmin=spec_features['vmin'])

    yellow_patch = patches.Patch(facecolor='yellow', edgecolor='k', label='Detections')

    legend_patches = [yellow_patch]
    ax = plt.gca()
    for i, row in plot_dets.iterrows():
        rect = patches.Rectangle(((row['start_time'] - start)*(fs/2), row['low_freq']/(fs/2)), 
                        (row['end_time'] - row['start_time'])*(fs/2), (row['high_freq'] - row['low_freq'])/(fs/2), 
                        linewidth=2, edgecolor=FREQUENCY_COLOR_MAPPINGS[row['KMEANS_CLASSES']], facecolor='none', alpha=1)
        
        ax.add_patch(rect)
    plt.yticks(ticks=np.linspace(0, 96000/(fs/2), 11), labels=np.linspace(0, 96, 11).astype('int'))
    plt.xticks(ticks=np.linspace(0, duration*(fs/2), 11), labels=np.round(np.linspace(start, start+duration, 11, dtype='float'), 2), rotation=30)
    plt.ylabel("Frequency (kHz)")
    plt.ylim(0, 96000/(fs/2))
    plt.xlabel("Time (s)")
    plt.gcf().autofmt_xdate()
    plt.grid(axis='y')
    plt.legend(handles=legend_patches, fontsize=20, ncol=int(len(legend_patches)**0.5), loc='upper right')

    plt.tight_layout()
    plt.show()


def plot_colored_dets_over_audio_seg_w_bounds(audio_features, spec_features, call_features, plot_dets):
    """
    Function to plot the spectrogram of a provided audio segment with overlayed detections
    """

    audio_seg = audio_features['audio_seg']
    fs = audio_features['sample_rate']
    start = audio_features['start']
    duration = audio_features['duration']

    plt.figure(figsize=FIGSIZE)
    plt.rcParams.update({'font.size': 24})
    plt.title(f"BatDetect2 detections on {audio_features['file_path'].name}", fontsize=22)
    plt.specgram(audio_seg, NFFT=spec_features['NFFT'], cmap=spec_features['cmap'], vmin=spec_features['vmin'])

    yellow_patch = patches.Patch(facecolor='yellow', edgecolor='k', label='Detections')

    legend_patches = [yellow_patch]
    ax = plt.gca()
    for i, row in plot_dets.iterrows():
        rect = patches.Rectangle(((row['start_time'] - start)*(fs/2), row['low_freq']/(fs/2)), 
                        (row['end_time'] - row['start_time'])*(fs/2), (row['high_freq'] - row['low_freq'])/(fs/2), 
                        linewidth=2, edgecolor=FREQUENCY_COLOR_MAPPINGS[row['KMEANS_CLASSES']], facecolor='none', alpha=1)
        
        ax.add_patch(rect)

    rect = patches.Rectangle((0,(call_features['median_lf']-7000)/(fs/2)), duration*fs, 14000/(fs/2), 
                             linewidth=2, edgecolor=FREQUENCY_COLOR_MAPPINGS['LF'], facecolor=FREQUENCY_COLOR_MAPPINGS['LF'], alpha=0.4)
    ax.add_patch(rect)
    rect = patches.Rectangle((0,(call_features['median_hf']-7000)/(fs/2)), duration*fs, (fs/2 - (call_features['median_hf']-7000))/(fs/2), 
                             linewidth=2, edgecolor=FREQUENCY_COLOR_MAPPINGS['HF'], facecolor=FREQUENCY_COLOR_MAPPINGS['HF'], alpha=0.4)
    ax.add_patch(rect)

    plt.yticks(ticks=np.linspace(0, 96000/(fs/2), 11), labels=np.linspace(0, 96, 11).astype('int'))
    plt.xticks(ticks=np.linspace(0, duration*(fs/2), 11), labels=np.round(np.linspace(start, start+duration, 11, dtype='float'), 2), rotation=30)
    plt.ylabel("Frequency (kHz)")
    plt.ylim(0, 96000/(fs/2))
    plt.xlabel("Time (s)")
    plt.gcf().autofmt_xdate()
    plt.grid(axis='y')
    plt.legend(handles=legend_patches, fontsize=20, ncol=int(len(legend_patches)**0.5), loc='upper right')

    plt.tight_layout()
    plt.show()


def plot_audio_seg(audio_features, spec_features):
    """
    Function to plot the spectrogram of a provided audio segment
    """

    audio_seg = audio_features['audio_seg']
    fs = audio_features['sample_rate']
    start = audio_features['start']
    duration = audio_features['duration']

    plt.figure(figsize=FIGSIZE)
    plt.rcParams.update({'font.size' : 24})
    plt.title(f"Audio collected from {audio_features['site_name']} on {audio_features['file_datetime']} UTC", fontsize=22)
    plt.rcParams.update({'font.size': 24})
    plt.specgram(audio_seg, NFFT=spec_features['NFFT'], cmap=spec_features['cmap'], vmin=spec_features['vmin'])
    plt.yticks(ticks=np.linspace(0, 96000/(fs/2), 11), labels=np.linspace(0, 96, 11).astype('int'))
    plt.xticks(ticks=np.linspace(0, duration*(fs/2), 11), labels=np.round(np.linspace(start, start+duration, 11, dtype='float'), 2), rotation=30)
    plt.ylabel("Frequency (kHz)")
    plt.ylim(0, 96000/(fs/2))
    plt.xlabel("Time (s)")
    plt.gcf().autofmt_xdate()
    plt.grid(axis='y')

    plt.tight_layout()
    plt.show()

def load_and_plot_all_examples_file(file_path, bd2_dets, start, duration, rm_dB=50, nfft=1024):
    audio_file = sf.SoundFile(file_path)
    fs = audio_file.samplerate
    audio_file.seek(int(fs*start))
    audio_seg = audio_file.read(int(fs*duration))
    vmin = 20*np.log10(np.max(audio_seg)) -  rm_dB # hide anything below -rm_dB dB

    audio_features = dict()
    audio_features['site_name'] = SITE_NAMES[file_path.parent.name]
    audio_features['file_datetime'] = dt.datetime.strptime(file_path.name, "%Y%m%d_%H%M%S.WAV").strftime('%Y/%m/%d %H:%M')
    audio_features['file_path'] = file_path
    audio_features['audio_seg'] = audio_seg
    audio_features['sample_rate'] = fs
    audio_features['start'] = start
    audio_features['duration'] = duration

    spec_features = dict()
    spec_features['vmin'] = vmin
    spec_features['NFFT'] = nfft
    spec_features['cmap'] = 'jet'

    site_key = file_path.parent.name

    plot_audio_seg(audio_features, spec_features)
    call_signals, dets = open_and_get_call_info(audio_file, bd2_dets.copy())
    plot_dets = dets.loc[(dets['start_time'] > start)&(dets['end_time'] < (start+duration))]
    plot_colored_dets_over_audio_seg(audio_features, spec_features, plot_dets)

    median_peak_HF_freq = dets[dets['KMEANS_CLASSES']=='HF']['peak_frequency'].median()
    median_peak_LF_freq = dets[dets['KMEANS_CLASSES']=='LF']['peak_frequency'].median()
    print(f'Median LF Frequency in file: {median_peak_LF_freq}')
    print(f'Median HF Frequency in file: {median_peak_HF_freq}')
    lf_inds = (plot_dets['peak_frequency']<median_peak_LF_freq+7000)&(plot_dets['peak_frequency']>median_peak_LF_freq-7000)
    hf_inds = (plot_dets['peak_frequency']>median_peak_HF_freq-7000)

    lf_dets = plot_dets[lf_inds&(plot_dets['KMEANS_CLASSES']=='LF')]
    hf_dets = plot_dets[hf_inds&(plot_dets['KMEANS_CLASSES']=='HF')]

    all_dets = pd.concat([hf_dets, lf_dets]).sort_index()

    call_features = dict()
    call_features['median_lf'] = median_peak_LF_freq
    call_features['median_hf'] = median_peak_HF_freq
    plot_colored_dets_over_audio_seg_w_bounds(audio_features, spec_features, call_features, all_dets)

    return plot_dets, all_dets

In [ ]:
site_key = 'Foliage'

fig_details = dict()
fig_details['site_name'] = SITE_NAMES[site_key]
print(f'Looking at {fig_details["site_name"]}')
files_from_loc = sorted(list(Path(f'../data/audiomoth_recordings/').glob(pattern=f'*/{site_key}/*.WAV')))

file_path = files_from_loc[0]
filename = file_path.name
csv_path = Path(f'../batdetect2_outputs/recover-20210912/{site_key}/bd2_{filename.split(".")[0]}.csv')
print(f'Looking at {file_path.name}')
start = 1439.5
duration = 1.5
rm_dB = 60
nfft = 512

data_params = dict()
data_params['site_tag'] = site_key
data_params['type_tag'] = ''
data_params['cur_dc_tag'] = '1800of1800'
batdetect2_predictions_no_dutycycle = actvt.assemble_single_bd2_output_use_thresholds_to_group(csv_path, data_params)

mis_dets, fix_dets = load_and_plot_all_examples_file(file_path, batdetect2_predictions_no_dutycycle, start, DURATION, rm_dB)

In [ ]:
removed_dets = mis_dets.loc[list(set(mis_dets.index) - set(fix_dets.index))]
removed_dets

In [ ]:
site_key = 'Foliage'

fig_details = dict()
fig_details['site_name'] = SITE_NAMES[site_key]
print(f'Looking at {fig_details["site_name"]}')
files_from_loc = sorted(list(Path(f'../data/audiomoth_recordings/').glob(pattern=f'*/{site_key}/*.WAV')))

file_path = files_from_loc[0]
filename = file_path.name
csv_path = Path(f'../batdetect2_outputs/recover-20210912/{site_key}/bd2_{filename.split(".")[0]}.csv')
print(f'Looking at {file_path.name}')
start = 1439.5
duration = 1.5
rm_dB = 60
nfft = 512

data_params = dict()
data_params['site_tag'] = site_key
data_params['type_tag'] = ''
data_params['cur_dc_tag'] = '1800of1800'
batdetect2_predictions_no_dutycycle = actvt.assemble_single_bd2_output_use_thresholds_to_group(csv_path, data_params)

mis_dets, fix_dets = load_and_plot_all_examples_file(file_path, batdetect2_predictions_no_dutycycle, start, duration, rm_dB)

In [10]:
removed_dets = mis_dets.loc[list(set(mis_dets.index) - set(fix_dets.index))]
removed_dets

,KMEANS_CLASSES,peak_frequency,SNR,freq_group,ref_time,call_start_time,call_end_time,start_time,end_time,low_freq,...,class_prob,det_prob,individual,input_file,Recover Folder,SD Card,Site name,index,file_name,sampling_rate
1953,HF,31680.0,23.218547,LF1,2021-09-10 03:24:00.371500,2021-09-10 03:24:00.371500,2021-09-10 03:24:00.379,1440.3715,1440.379,27187.0,...,0.302,0.53,-1,/Users/adityakrishna/Documents/Research/Lab_re...,recover-20210912_unit2,UBNA_002,Foliage,1953,20210910_030000.WAV,250000


In [ ]:
site_key = 'Foliage'

fig_details = dict()
fig_details['site_name'] = SITE_NAMES[site_key]
print(f'Looking at {fig_details["site_name"]}')
files_from_loc = sorted(list(Path(f'../data/audiomoth_recordings/').glob(pattern=f'*/{site_key}/*.WAV')))

file_path = files_from_loc[0]
filename = file_path.name
csv_path = Path(f'../batdetect2_outputs/recover-20210912/{site_key}/bd2_{filename.split(".")[0]}.csv')
print(f'Looking at {file_path.name}')
start = 1459
duration = 1
rm_dB = 60
nfft = 512

data_params = dict()
data_params['site_tag'] = site_key
data_params['type_tag'] = ''
data_params['cur_dc_tag'] = '1800of1800'
batdetect2_predictions_no_dutycycle = actvt.assemble_single_bd2_output_use_thresholds_to_group(csv_path, data_params)

mis_dets, fix_dets = load_and_plot_all_examples_file(file_path, batdetect2_predictions_no_dutycycle, start, duration, rm_dB)

In [12]:
list(set(mis_dets.index) - set(fix_dets.index))

[2112]

In [13]:
removed_dets = mis_dets.loc[list(set(mis_dets.index) - set(fix_dets.index))]
removed_dets

,KMEANS_CLASSES,peak_frequency,SNR,freq_group,ref_time,call_start_time,call_end_time,start_time,end_time,low_freq,...,class_prob,det_prob,individual,input_file,Recover Folder,SD Card,Site name,index,file_name,sampling_rate
2112,LF,34560.0,5.338977,LF1,2021-09-10 03:24:19.342500,2021-09-10 03:24:19.342500,2021-09-10 03:24:19.348400,1459.3425,1459.3484,28906.0,...,0.466,0.573,-1,/Users/adityakrishna/Documents/Research/Lab_re...,recover-20210912_unit2,UBNA_002,Foliage,2112,20210910_030000.WAV,250000


In [14]:
data_params["site_name"] = 'Foliage'
data_params["site_tag"] = 'Foliage'
data_params["type_tag"] = ''
data_params["detector_tag"] = 'bd2'
data_params["assembly_type"] = 'thresh'

file_paths = get_file_paths(data_params)

location_df_thresh = pd.read_csv(f'{file_paths["SITE_folder"]}/{file_paths["detector_TYPE_SITE_YEAR"]}.csv', low_memory=False, index_col=0)
location_df_thresh

,freq_group,index_in_file,ref_time,call_start_time,call_end_time,start_time,end_time,low_freq,high_freq,event,class,class_prob,det_prob,individual,input_file,Site name,Recover Folder,SD Card,File Duration
0,LF1,0,2022-06-15 03:33:24.469500000,2022-06-15 03:33:24.469500000,2022-06-15 03:33:24.485600000,204.4695,204.4856,18593.0,23399.0,Echolocation,Nyctalus noctula,0.399,0.544,-1,/mnt/ubna_data_01/recover-20220616_unit2/20220...,Foliage,recover-20220616_unit2,10,NaN
1,HF1,0,2022-06-15 04:32:25.191500000,2022-06-15 04:32:25.191500000,2022-06-15 04:32:25.197100000,145.1915,145.1971,41796.0,61385.0,Echolocation,Pipistrellus nathusii,0.364,0.512,-1,/mnt/ubna_data_01/recover-20220616_unit2/20220...,Foliage,recover-20220616_unit2,10,NaN
2,HF1,1,2022-06-15 04:32:25.625500000,2022-06-15 04:32:25.625500000,2022-06-15 04:32:25.632500000,145.6255,145.6325,40937.0,63845.0,Echolocation,Pipistrellus nathusii,0.488,0.543,-1,/mnt/ubna_data_01/recover-20220616_unit2/20220...,Foliage,recover-20220616_unit2,10,NaN
3,HF1,2,2022-06-15 04:32:25.797500000,2022-06-15 04:32:25.797500000,2022-06-15 04:32:25.803800000,145.7975,145.8038,41796.0,61714.0,Echolocation,Pipistrellus nathusii,0.473,0.513,-1,/mnt/ubna_data_01/recover-20220616_unit2/20220...,Foliage,recover-20220616_unit2,10,NaN
4,HF1,3,2022-06-15 04:32:26.110500000,2022-06-15 04:32:26.110500000,2022-06-15 04:32:26.115800000,146.1105,146.1158,35781.0,62458.0,Echolocation,Myotis brandtii,0.381,0.600,-1,/mnt/ubna_data_01/recover-20220616_unit2/20220...,Foliage,recover-20220616_unit2,10,NaN
...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...
645606,HF2,9,2022-10-17 08:32:31.083500000,2022-10-17 08:32:31.083500000,2022-10-17 08:32:31.095200000,151.0835,151.0952,45234.0,62435.0,Echolocation,Pipistrellus pipistrellus,0.463,0.548,-1,/mnt/ubna_data_02/recover-20221017/UBNA_008/20...,Foliage,recover-20221017,8,NaN
645607,HF2,10,2022-10-17 08:32:31.488500000,2022-10-17 08:32:31.488500000,2022-10-17 08:32:31.494600000,151.4885,151.4946,45234.0,55595.0,Echolocation,Pipistrellus pipistrellus,0.612,0.637,-1,/mnt/ubna_data_02/recover-20221017/UBNA_008/20...,Foliage,recover-20221017,8,NaN
645608,HF2,11,2022-10-17 08:32:32.354500000,2022-10-17 08:32:32.354500000,2022-10-17 08:32:32.360600000,152.3545,152.3606,46953.0,61784.0,Echolocation,Pipistrellus pipistrellus,0.389,0.505,-1,/mnt/ubna_data_02/recover-20221017/UBNA_008/20...,Foliage,recover-20221017,8,NaN
645609,HF2,0,2022-10-17 10:21:52.593500000,2022-10-17 10:21:52.593500000,2022-10-17 10:21:52.600200000,1312.5935,1312.6002,46953.0,57745.0,Echolocation,Pipistrellus pipistrellus,0.494,0.533,-1,/mnt/ubna_data_02/recover-20221017/UBNA_008/20...,Foliage,recover-20221017,8,NaN


In [15]:
selection_date = dt.datetime(2022,6,15,5,0,0)

In [16]:
selected_group_thresh = location_df_thresh[pd.to_datetime(location_df_thresh['input_file'], format="%Y%m%d_%H%M%S.WAV", exact=False)<=selection_date]
selected_group_thresh

,freq_group,index_in_file,ref_time,call_start_time,call_end_time,start_time,end_time,low_freq,high_freq,event,class,class_prob,det_prob,individual,input_file,Site name,Recover Folder,SD Card,File Duration
0,LF1,0,2022-06-15 03:33:24.469500000,2022-06-15 03:33:24.469500000,2022-06-15 03:33:24.485600000,204.4695,204.4856,18593.0,23399.0,Echolocation,Nyctalus noctula,0.399,0.544,-1,/mnt/ubna_data_01/recover-20220616_unit2/20220...,Foliage,recover-20220616_unit2,10,NaN
1,HF1,0,2022-06-15 04:32:25.191500000,2022-06-15 04:32:25.191500000,2022-06-15 04:32:25.197100000,145.1915,145.1971,41796.0,61385.0,Echolocation,Pipistrellus nathusii,0.364,0.512,-1,/mnt/ubna_data_01/recover-20220616_unit2/20220...,Foliage,recover-20220616_unit2,10,NaN
2,HF1,1,2022-06-15 04:32:25.625500000,2022-06-15 04:32:25.625500000,2022-06-15 04:32:25.632500000,145.6255,145.6325,40937.0,63845.0,Echolocation,Pipistrellus nathusii,0.488,0.543,-1,/mnt/ubna_data_01/recover-20220616_unit2/20220...,Foliage,recover-20220616_unit2,10,NaN
3,HF1,2,2022-06-15 04:32:25.797500000,2022-06-15 04:32:25.797500000,2022-06-15 04:32:25.803800000,145.7975,145.8038,41796.0,61714.0,Echolocation,Pipistrellus nathusii,0.473,0.513,-1,/mnt/ubna_data_01/recover-20220616_unit2/20220...,Foliage,recover-20220616_unit2,10,NaN
4,HF1,3,2022-06-15 04:32:26.110500000,2022-06-15 04:32:26.110500000,2022-06-15 04:32:26.115800000,146.1105,146.1158,35781.0,62458.0,Echolocation,Myotis brandtii,0.381,0.600,-1,/mnt/ubna_data_01/recover-20220616_unit2/20220...,Foliage,recover-20220616_unit2,10,NaN
...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...
1850,LF1,377,2022-06-15 05:29:13.351500000,2022-06-15 05:29:13.351500000,2022-06-15 05:29:13.366600000,1753.3515,1753.3666,24609.0,27983.0,Echolocation,Nyctalus leisleri,0.532,0.661,-1,/mnt/ubna_data_01/recover-20220616_unit2/20220...,Foliage,recover-20220616_unit2,10,NaN
1851,LF1,378,2022-06-15 05:29:13.529500000,2022-06-15 05:29:13.529500000,2022-06-15 05:29:13.543100000,1753.5295,1753.5431,24609.0,28725.0,Echolocation,Nyctalus leisleri,0.516,0.655,-1,/mnt/ubna_data_01/recover-20220616_unit2/20220...,Foliage,recover-20220616_unit2,10,NaN
1852,LF1,379,2022-06-15 05:29:13.818500000,2022-06-15 05:29:13.818500000,2022-06-15 05:29:13.832800000,1753.8185,1753.8328,24609.0,28370.0,Echolocation,Nyctalus leisleri,0.575,0.668,-1,/mnt/ubna_data_01/recover-20220616_unit2/20220...,Foliage,recover-20220616_unit2,10,NaN
1853,LF1,380,2022-06-15 05:29:16.061500000,2022-06-15 05:29:16.061500000,2022-06-15 05:29:16.077100000,1756.0615,1756.0771,24609.0,28619.0,Echolocation,Nyctalus leisleri,0.532,0.647,-1,/mnt/ubna_data_01/recover-20220616_unit2/20220...,Foliage,recover-20220616_unit2,10,NaN


In [17]:
def add_frequency_group_to_file_dets(file_dets, location_classes):
    file_classes = location_classes[pd.to_datetime(location_classes['file_name'], 
                                                   format='%Y%m%d_%H%M%S.WAV', exact=False)==file_dets.name].copy()

    file_dets.insert(0, 'index_in_summary', file_dets.index)
    file_dets.set_index('index_in_file', inplace=True)

    classified = file_classes['KMEANS_CLASSES']!=''
    file_classes.loc[classified, 'peak_frequency'] = file_classes.loc[classified, 'peak_frequency'].astype('float64')

    file_dets.insert(0, 'peak_frequency', [np.NaN]*len(file_dets))
    file_dets.loc[file_classes['index_in_file'], 'freq_group'] = file_classes['KMEANS_CLASSES'].values
    file_dets.loc[file_classes['index_in_file'], 'peak_frequency'] = file_classes['peak_frequency'].values

    # for group in ['LF', 'HF']:
    #     group_classified_dets = (file_dets['freq_group']==group)

    #     low_assert1 = (file_dets.loc[group_classified_dets, 'peak_frequency'] > (file_dets.loc[group_classified_dets, 'low_freq']).median()-4000)
    #     low_assert2 = (file_dets.loc[group_classified_dets, 'peak_frequency'] > (file_dets.loc[group_classified_dets, 'low_freq'])-4000)
    #     assert(low_assert1|low_assert2).all()
    #     high_assert1 = (file_dets.loc[group_classified_dets, 'peak_frequency'] < (file_dets.loc[group_classified_dets, 'high_freq']).median()+4000)
    #     high_assert2 = (file_dets.loc[group_classified_dets, 'peak_frequency'] < (file_dets.loc[group_classified_dets, 'high_freq'])+4000)
    #     assert(high_assert1|high_assert2).all()

    return file_dets

def add_frequency_groups_to_summary_using_kmeans(location_df, file_paths, data_params, save=True):
    location_df.insert(0, 'freq_group', '')
    location_classes = pd.read_csv(Path(file_paths['SITE_classes_file']), index_col=0)
    location_df.insert(0, 'input_file_dt', pd.to_datetime(location_df['input_file'], format='%Y%m%d_%H%M%S.WAV', exact=False))
    location_df_grouped = location_df.groupby('input_file_dt', group_keys=True)

    location_df_classified = location_df_grouped.apply(lambda x: add_frequency_group_to_file_dets(x, location_classes))

    location_df_only_classified = location_df_classified.loc[location_df_classified['freq_group']!='']
    location_df_only_classified = location_df_only_classified.droplevel(level=0)
    location_df_only_classified = location_df_only_classified.reset_index()

    if data_params['type_tag'] != '':
        location_df_only_classified = location_df_only_classified.loc[location_df_only_classified['freq_group']==data_params['type_tag']]

    if save:
        location_df_only_classified.to_csv(f'{file_paths["SITE_folder"]}/{file_paths["detector_TYPE_SITE_YEAR"]}.csv')

    return location_df_only_classified

In [18]:
data_params["site_name"] = 'Foliage'
data_params["site_tag"] = 'Foliage'
data_params["type_tag"] = ''
data_params["detector_tag"] = 'bd2'
data_params["assembly_type"] = 'kmeans'

file_paths = get_file_paths(data_params)
file_paths['SITE_classes_file'] = f"{file_paths['SITE_classes_file'][:-4]}_raw.csv"

init_location_sum = actvt.assemble_initial_location_summary(file_paths) 
init_location_sum.reset_index(inplace=True)
init_location_sum.rename({'index':'index_in_file'}, axis='columns', inplace=True)
location_df_kmeans_raw = add_frequency_groups_to_summary_using_kmeans(init_location_sum.copy(), file_paths, data_params, save=False)

In [19]:
location_df_kmeans_raw

,index_in_file,peak_frequency,index_in_summary,input_file_dt,freq_group,ref_time,call_start_time,call_end_time,start_time,end_time,...,event,class,class_prob,det_prob,individual,input_file,Site name,Recover Folder,SD Card,File Duration
0,0,20160.0,0,2022-06-15 03:30:00,LF,2022-06-15 03:33:24.469500,2022-06-15 03:33:24.469500,2022-06-15 03:33:24.485600,204.4695,204.4856,...,Echolocation,Nyctalus noctula,0.399,0.544,-1,/mnt/ubna_data_01/recover-20220616_unit2/20220...,Foliage,recover-20220616_unit2,10,NaN
1,0,47040.0,1,2022-06-15 04:30:00,HF,2022-06-15 04:32:25.191500,2022-06-15 04:32:25.191500,2022-06-15 04:32:25.197100,145.1915,145.1971,...,Echolocation,Pipistrellus nathusii,0.364,0.512,-1,/mnt/ubna_data_01/recover-20220616_unit2/20220...,Foliage,recover-20220616_unit2,10,NaN
2,1,44160.0,2,2022-06-15 04:30:00,HF,2022-06-15 04:32:25.625500,2022-06-15 04:32:25.625500,2022-06-15 04:32:25.632500,145.6255,145.6325,...,Echolocation,Pipistrellus nathusii,0.488,0.543,-1,/mnt/ubna_data_01/recover-20220616_unit2/20220...,Foliage,recover-20220616_unit2,10,NaN
3,2,45120.0,3,2022-06-15 04:30:00,HF,2022-06-15 04:32:25.797500,2022-06-15 04:32:25.797500,2022-06-15 04:32:25.803800,145.7975,145.8038,...,Echolocation,Pipistrellus nathusii,0.473,0.513,-1,/mnt/ubna_data_01/recover-20220616_unit2/20220...,Foliage,recover-20220616_unit2,10,NaN
4,3,45120.0,4,2022-06-15 04:30:00,HF,2022-06-15 04:32:26.110500,2022-06-15 04:32:26.110500,2022-06-15 04:32:26.115800,146.1105,146.1158,...,Echolocation,Myotis brandtii,0.381,0.6,-1,/mnt/ubna_data_01/recover-20220616_unit2/20220...,Foliage,recover-20220616_unit2,10,NaN
...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...
628663,9,46080.0,645606,2022-10-17 08:30:00,HF,2022-10-17 08:32:31.083500,2022-10-17 08:32:31.083500,2022-10-17 08:32:31.095200,151.0835,151.0952,...,Echolocation,Pipistrellus pipistrellus,0.463,0.548,-1,/mnt/ubna_data_02/recover-20221017/UBNA_008/20...,Foliage,recover-20221017,8,NaN
628664,10,46080.0,645607,2022-10-17 08:30:00,HF,2022-10-17 08:32:31.488500,2022-10-17 08:32:31.488500,2022-10-17 08:32:31.494600,151.4885,151.4946,...,Echolocation,Pipistrellus pipistrellus,0.612,0.637,-1,/mnt/ubna_data_02/recover-20221017/UBNA_008/20...,Foliage,recover-20221017,8,NaN
628665,11,48960.0,645608,2022-10-17 08:30:00,HF,2022-10-17 08:32:32.354500,2022-10-17 08:32:32.354500,2022-10-17 08:32:32.360600,152.3545,152.3606,...,Echolocation,Pipistrellus pipistrellus,0.389,0.505,-1,/mnt/ubna_data_02/recover-20221017/UBNA_008/20...,Foliage,recover-20221017,8,NaN
628666,0,48000.0,645609,2022-10-17 10:00:00,HF,2022-10-17 10:21:52.593500,2022-10-17 10:21:52.593500,2022-10-17 10:21:52.600200,1312.5935,1312.6002,...,Echolocation,Pipistrellus pipistrellus,0.494,0.533,-1,/mnt/ubna_data_02/recover-20221017/UBNA_008/20...,Foliage,recover-20221017,8,NaN


In [20]:
data_params["site_name"] = 'Foliage'
data_params["site_tag"] = 'Foliage'
data_params["type_tag"] = ''
data_params["detector_tag"] = 'bd2'
data_params["assembly_type"] = 'kmeans'

file_paths = get_file_paths(data_params)

location_df_kmeans = pd.read_csv(f'{file_paths["SITE_folder"]}/{file_paths["detector_TYPE_SITE_YEAR"]}.csv', low_memory=False, index_col=0)
location_df_kmeans

,index_in_file,peak_frequency,index_in_summary,input_file_dt,freq_group,ref_time,call_start_time,call_end_time,start_time,end_time,...,event,class,class_prob,det_prob,individual,input_file,Site name,Recover Folder,SD Card,File Duration
0,0,20160.0,0,2022-06-15 03:30:00,LF,2022-06-15 03:33:24.469500000,2022-06-15 03:33:24.469500000,2022-06-15 03:33:24.485600000,204.4695,204.4856,...,Echolocation,Nyctalus noctula,0.399,0.544,-1,/mnt/ubna_data_01/recover-20220616_unit2/20220...,Foliage,recover-20220616_unit2,10,NaN
1,0,47040.0,1,2022-06-15 04:30:00,HF,2022-06-15 04:32:25.191500000,2022-06-15 04:32:25.191500000,2022-06-15 04:32:25.197100000,145.1915,145.1971,...,Echolocation,Pipistrellus nathusii,0.364,0.512,-1,/mnt/ubna_data_01/recover-20220616_unit2/20220...,Foliage,recover-20220616_unit2,10,NaN
2,1,44160.0,2,2022-06-15 04:30:00,HF,2022-06-15 04:32:25.625500000,2022-06-15 04:32:25.625500000,2022-06-15 04:32:25.632500000,145.6255,145.6325,...,Echolocation,Pipistrellus nathusii,0.488,0.543,-1,/mnt/ubna_data_01/recover-20220616_unit2/20220...,Foliage,recover-20220616_unit2,10,NaN
3,2,45120.0,3,2022-06-15 04:30:00,HF,2022-06-15 04:32:25.797500000,2022-06-15 04:32:25.797500000,2022-06-15 04:32:25.803800000,145.7975,145.8038,...,Echolocation,Pipistrellus nathusii,0.473,0.513,-1,/mnt/ubna_data_01/recover-20220616_unit2/20220...,Foliage,recover-20220616_unit2,10,NaN
4,3,45120.0,4,2022-06-15 04:30:00,HF,2022-06-15 04:32:26.110500000,2022-06-15 04:32:26.110500000,2022-06-15 04:32:26.115800000,146.1105,146.1158,...,Echolocation,Myotis brandtii,0.381,0.600,-1,/mnt/ubna_data_01/recover-20220616_unit2/20220...,Foliage,recover-20220616_unit2,10,NaN
...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...
626740,9,46080.0,645606,2022-10-17 08:30:00,HF,2022-10-17 08:32:31.083500000,2022-10-17 08:32:31.083500000,2022-10-17 08:32:31.095200000,151.0835,151.0952,...,Echolocation,Pipistrellus pipistrellus,0.463,0.548,-1,/mnt/ubna_data_02/recover-20221017/UBNA_008/20...,Foliage,recover-20221017,8,NaN
626741,10,46080.0,645607,2022-10-17 08:30:00,HF,2022-10-17 08:32:31.488500000,2022-10-17 08:32:31.488500000,2022-10-17 08:32:31.494600000,151.4885,151.4946,...,Echolocation,Pipistrellus pipistrellus,0.612,0.637,-1,/mnt/ubna_data_02/recover-20221017/UBNA_008/20...,Foliage,recover-20221017,8,NaN
626742,11,48960.0,645608,2022-10-17 08:30:00,HF,2022-10-17 08:32:32.354500000,2022-10-17 08:32:32.354500000,2022-10-17 08:32:32.360600000,152.3545,152.3606,...,Echolocation,Pipistrellus pipistrellus,0.389,0.505,-1,/mnt/ubna_data_02/recover-20221017/UBNA_008/20...,Foliage,recover-20221017,8,NaN
626743,0,48000.0,645609,2022-10-17 10:00:00,HF,2022-10-17 10:21:52.593500000,2022-10-17 10:21:52.593500000,2022-10-17 10:21:52.600200000,1312.5935,1312.6002,...,Echolocation,Pipistrellus pipistrellus,0.494,0.533,-1,/mnt/ubna_data_02/recover-20221017/UBNA_008/20...,Foliage,recover-20221017,8,NaN


In [21]:
selected_group_kmeans = location_df_kmeans[pd.to_datetime(location_df_kmeans['input_file'], format="%Y%m%d_%H%M%S.WAV", exact=False)<=selection_date]
selected_group_kmeans

,index_in_file,peak_frequency,index_in_summary,input_file_dt,freq_group,ref_time,call_start_time,call_end_time,start_time,end_time,...,event,class,class_prob,det_prob,individual,input_file,Site name,Recover Folder,SD Card,File Duration
0,0,20160.0,0,2022-06-15 03:30:00,LF,2022-06-15 03:33:24.469500000,2022-06-15 03:33:24.469500000,2022-06-15 03:33:24.485600000,204.4695,204.4856,...,Echolocation,Nyctalus noctula,0.399,0.544,-1,/mnt/ubna_data_01/recover-20220616_unit2/20220...,Foliage,recover-20220616_unit2,10,NaN
1,0,47040.0,1,2022-06-15 04:30:00,HF,2022-06-15 04:32:25.191500000,2022-06-15 04:32:25.191500000,2022-06-15 04:32:25.197100000,145.1915,145.1971,...,Echolocation,Pipistrellus nathusii,0.364,0.512,-1,/mnt/ubna_data_01/recover-20220616_unit2/20220...,Foliage,recover-20220616_unit2,10,NaN
2,1,44160.0,2,2022-06-15 04:30:00,HF,2022-06-15 04:32:25.625500000,2022-06-15 04:32:25.625500000,2022-06-15 04:32:25.632500000,145.6255,145.6325,...,Echolocation,Pipistrellus nathusii,0.488,0.543,-1,/mnt/ubna_data_01/recover-20220616_unit2/20220...,Foliage,recover-20220616_unit2,10,NaN
3,2,45120.0,3,2022-06-15 04:30:00,HF,2022-06-15 04:32:25.797500000,2022-06-15 04:32:25.797500000,2022-06-15 04:32:25.803800000,145.7975,145.8038,...,Echolocation,Pipistrellus nathusii,0.473,0.513,-1,/mnt/ubna_data_01/recover-20220616_unit2/20220...,Foliage,recover-20220616_unit2,10,NaN
4,3,45120.0,4,2022-06-15 04:30:00,HF,2022-06-15 04:32:26.110500000,2022-06-15 04:32:26.110500000,2022-06-15 04:32:26.115800000,146.1105,146.1158,...,Echolocation,Myotis brandtii,0.381,0.600,-1,/mnt/ubna_data_01/recover-20220616_unit2/20220...,Foliage,recover-20220616_unit2,10,NaN
...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...
1792,377,25920.0,1850,2022-06-15 05:00:00,LF,2022-06-15 05:29:13.351500000,2022-06-15 05:29:13.351500000,2022-06-15 05:29:13.366600000,1753.3515,1753.3666,...,Echolocation,Nyctalus leisleri,0.532,0.661,-1,/mnt/ubna_data_01/recover-20220616_unit2/20220...,Foliage,recover-20220616_unit2,10,NaN
1793,378,25920.0,1851,2022-06-15 05:00:00,LF,2022-06-15 05:29:13.529500000,2022-06-15 05:29:13.529500000,2022-06-15 05:29:13.543100000,1753.5295,1753.5431,...,Echolocation,Nyctalus leisleri,0.516,0.655,-1,/mnt/ubna_data_01/recover-20220616_unit2/20220...,Foliage,recover-20220616_unit2,10,NaN
1794,379,25920.0,1852,2022-06-15 05:00:00,LF,2022-06-15 05:29:13.818500000,2022-06-15 05:29:13.818500000,2022-06-15 05:29:13.832800000,1753.8185,1753.8328,...,Echolocation,Nyctalus leisleri,0.575,0.668,-1,/mnt/ubna_data_01/recover-20220616_unit2/20220...,Foliage,recover-20220616_unit2,10,NaN
1795,380,25920.0,1853,2022-06-15 05:00:00,LF,2022-06-15 05:29:16.061500000,2022-06-15 05:29:16.061500000,2022-06-15 05:29:16.077100000,1756.0615,1756.0771,...,Echolocation,Nyctalus leisleri,0.532,0.647,-1,/mnt/ubna_data_01/recover-20220616_unit2/20220...,Foliage,recover-20220616_unit2,10,NaN


In [22]:
def get_dropped_by_kmeans(thresh_file_df, all_file_kmeans_df):
    input_file_group_name = thresh_file_df.input_file.values[0]
    thresh_file_df = thresh_file_df.set_index('index_in_file')
    kmeans_file_df = all_file_kmeans_df[all_file_kmeans_df['input_file']==input_file_group_name]
    kmeans_file_df = kmeans_file_df.set_index('index_in_file')
    dropped_inds = sorted(list(set(thresh_file_df.index) - set(kmeans_file_df.index)))
    return thresh_file_df.loc[dropped_inds]

In [23]:
# all_dropped_calls = location_sum_kmeans.groupby(by='input_file', group_keys=False).apply(lambda x : get_dropped_by_median_freq_removal(x, location_sum_kmeans_remove_fp))
all_dropped_calls = location_df_kmeans_raw.groupby(by='input_file', group_keys=False).apply(lambda x : get_dropped_by_kmeans(x, location_df_kmeans))

In [24]:
all_dropped_calls

,peak_frequency,index_in_summary,input_file_dt,freq_group,ref_time,call_start_time,call_end_time,start_time,end_time,low_freq,...,event,class,class_prob,det_prob,individual,input_file,Site name,Recover Folder,SD Card,File Duration
index_in_file,,,,,,,,,,,,,,,,,,,,,
372,31680.0,373,2022-06-15 04:30:00,HF,2022-06-15 04:43:09.327500,2022-06-15 04:43:09.327500,2022-06-15 04:43:09.334800000,789.3275,789.3348,28046.0,...,Echolocation,Eptesicus serotinus,0.365,0.723,-1,/mnt/ubna_data_01/recover-20220616_unit2/20220...,Foliage,recover-20220616_unit2,10,NaN
373,33600.0,374,2022-06-15 04:30:00,HF,2022-06-15 04:43:09.403500,2022-06-15 04:43:09.403500,2022-06-15 04:43:09.409400000,789.4035,789.4094,28046.0,...,Echolocation,Eptesicus serotinus,0.493,0.671,-1,/mnt/ubna_data_01/recover-20220616_unit2/20220...,Foliage,recover-20220616_unit2,10,NaN
376,27840.0,377,2022-06-15 04:30:00,HF,2022-06-15 04:43:10.425500,2022-06-15 04:43:10.425500,2022-06-15 04:43:10.435100000,790.4255,790.4351,26328.0,...,Echolocation,Eptesicus serotinus,0.561,0.696,-1,/mnt/ubna_data_01/recover-20220616_unit2/20220...,Foliage,recover-20220616_unit2,10,NaN
377,29760.0,378,2022-06-15 04:30:00,HF,2022-06-15 04:43:10.565500,2022-06-15 04:43:10.565500,2022-06-15 04:43:10.573500000,790.5655,790.5735,27187.0,...,Echolocation,Eptesicus serotinus,0.569,0.735,-1,/mnt/ubna_data_01/recover-20220616_unit2/20220...,Foliage,recover-20220616_unit2,10,NaN
380,29760.0,381,2022-06-15 04:30:00,HF,2022-06-15 04:43:10.957500,2022-06-15 04:43:10.957500,2022-06-15 04:43:10.966000000,790.9575,790.9660,27187.0,...,Echolocation,Eptesicus serotinus,0.534,0.704,-1,/mnt/ubna_data_01/recover-20220616_unit2/20220...,Foliage,recover-20220616_unit2,10,NaN
...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...
6,44160.0,637989,2022-10-08 10:30:00,HF,2022-10-08 10:33:07.759500,2022-10-08 10:33:07.759500,2022-10-08 10:33:07.767500000,187.7595,187.7675,40078.0,...,Echolocation,Pipistrellus nathusii,0.428,0.526,-1,/mnt/ubna_data_02/recover-20221010/UBNA_008/20...,Foliage,recover-20221010,8,NaN
7,42240.0,637990,2022-10-08 10:30:00,HF,2022-10-08 10:33:07.858500,2022-10-08 10:33:07.858500,2022-10-08 10:33:07.865600000,187.8585,187.8656,40937.0,...,Echolocation,Pipistrellus nathusii,0.527,0.564,-1,/mnt/ubna_data_02/recover-20221010/UBNA_008/20...,Foliage,recover-20221010,8,NaN
8,44160.0,637991,2022-10-08 10:30:00,HF,2022-10-08 10:33:07.958500,2022-10-08 10:33:07.958500,2022-10-08 10:33:07.966000000,187.9585,187.9660,40078.0,...,Echolocation,Pipistrellus nathusii,0.585,0.653,-1,/mnt/ubna_data_02/recover-20221010/UBNA_008/20...,Foliage,recover-20221010,8,NaN


In [25]:
test_df = all_dropped_calls.reset_index().loc[:,['index_in_file', 'freq_group', 'start_time', 'end_time', 'low_freq', 'high_freq', 'input_file']]
test_df

,index_in_file,freq_group,start_time,end_time,low_freq,high_freq,input_file
0,372,HF,789.3275,789.3348,28046.0,47777.0,/mnt/ubna_data_01/recover-20220616_unit2/20220...
1,373,HF,789.4035,789.4094,28046.0,48337.0,/mnt/ubna_data_01/recover-20220616_unit2/20220...
2,376,HF,790.4255,790.4351,26328.0,46437.0,/mnt/ubna_data_01/recover-20220616_unit2/20220...
3,377,HF,790.5655,790.5735,27187.0,47151.0,/mnt/ubna_data_01/recover-20220616_unit2/20220...
4,380,HF,790.9575,790.9660,27187.0,47882.0,/mnt/ubna_data_01/recover-20220616_unit2/20220...
...,...,...,...,...,...,...,...
1918,6,HF,187.7595,187.7675,40078.0,57041.0,/mnt/ubna_data_02/recover-20221010/UBNA_008/20...
1919,7,HF,187.8585,187.8656,40937.0,47775.0,/mnt/ubna_data_02/recover-20221010/UBNA_008/20...
1920,8,HF,187.9585,187.9660,40078.0,54172.0,/mnt/ubna_data_02/recover-20221010/UBNA_008/20...
1921,224,HF,291.2595,291.2673,35781.0,44090.0,/mnt/ubna_data_02/recover-20221017/UBNA_008/20...


In [26]:
def get_section_of_call_in_file(detection, audio_file):
    fs = audio_file.samplerate

    call_dur = (detection['end_time'] - detection['start_time'])
    pad = min(min(detection['start_time'] - call_dur, 1795 - detection['end_time']), 0.006) / 3
    start = detection['start_time'] - call_dur - (3*pad)
    duration = (2 * call_dur) + (4*pad)

    audio_file.seek(int(fs*start))
    audio_seg = audio_file.read(int(fs*duration))

    length_of_section = call_dur + (2*pad)

    return audio_seg, length_of_section

In [27]:
filesys = fsspec.filesystem('s3', anon=True, client_kwargs={'endpoint_url': 'https://sdsc.osn.xsede.org'})

In [28]:
FREQUENCY_COLOR_MAPPINGS = {
                    'LF' : 'cyan',
                    'HF' : 'orange'
                        }

In [29]:
import re

In [ ]:
cur_path = ''
for i in np.arange(0, len(test_df), 36):
    subset = test_df[i:min(i+36, len(test_df))].reset_index(drop=True).copy()
    matrix_side_length = int(np.ceil(len(subset)**0.5))
    plt.figure(figsize=(3*matrix_side_length,3*matrix_side_length))
    plt.rcParams.update({'font.size':12})

    for j, row in subset.iterrows():
        plt.subplot(matrix_side_length, matrix_side_length, j+1)
        file_path = '/'.join(Path(row['input_file']).parts[2:])
        cleaned_path = re.sub(r"(ubna_data_\d+)_mir", r"\1", file_path)
        osn_file_path = Path(f'bio230143-bucket01/{cleaned_path}')
        if cur_path!=osn_file_path:
            cur_path = osn_file_path
            file = filesys.open(path=cur_path)
            audio_file = sf.SoundFile(file)
            fs = audio_file.samplerate

        # audio_seg, length_of_section = get_section_of_call_in_file(row, audio_file)
        call_dur = (row['end_time'] - row['start_time'])
        pad = min(min(row['start_time'] - call_dur, 1795 - row['end_time']), 0.3) / 3
        start = row['start_time'] - call_dur - (3*pad)
        duration = (2 * call_dur) + (10*pad)

        audio_file.seek(int(fs*start))
        audio_seg = audio_file.read(int(fs*duration))

        plt.title(f'{osn_file_path.name}')
        plt.specgram(audio_seg, NFFT=256, cmap='jet', vmin=-60, vmax=0)

        file_df_orig = location_df_kmeans_raw[location_df_kmeans_raw['input_file']==row['input_file']]
        plot_dets = file_df_orig[(file_df_orig['start_time']>=start)&(file_df_orig['end_time']<=(start+duration))]
        ax = plt.gca()
        for k, det in plot_dets.iterrows():
            if det['start_time']==row['start_time']:
                rect = patches.Rectangle(((det['start_time'] - start)*(fs/2), det['low_freq']/(fs/2)), 
                                (det['end_time'] - det['start_time'])*(fs/2), (det['high_freq'] - det['low_freq'])/(fs/2), 
                                linewidth=3, edgecolor='red', facecolor='none', alpha=0.8)
            else:
                rect = patches.Rectangle(((det['start_time'] - start)*(fs/2), det['low_freq']/(fs/2)), 
                                (det['end_time'] - det['start_time'])*(fs/2), (det['high_freq'] - det['low_freq'])/(fs/2), 
                                linewidth=2, edgecolor=FREQUENCY_COLOR_MAPPINGS[det['freq_group']], facecolor='none', alpha=0.8)
            ax.add_patch(rect)

        plt.yticks(ticks=np.linspace(0, 1, 6), labels=np.linspace(0, fs/2000, 6).astype('int'))
        plt.ylabel("Frequency (kHz)", fontsize=14)
        plt.text(x=int(fs*0.001),y=0.85, s=f'{row["freq_group"]} det{i+j}', fontweight='bold', color='w')
        plt.xticks(ticks=np.linspace(0, duration*fs/2, 6), labels=np.round(np.linspace(start, start+duration, 6, dtype=float), 3), rotation=30)
        plt.xlabel("Time (s)")

    plt.tight_layout()
    plt.show()